In [ ]:
import scanpy as sc
import cell2location
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import scipy.sparse
import pyro.optim
from cell2location.plt import plot_spatial
from lightning.pytorch.callbacks import EarlyStopping

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU")


def preprocess_data(adata):
    sc.pp.filter_genes(adata, min_cells=10)
    adata.var_names_make_unique()
    return adata


def load_and_split_data(visium_path, sc_path):
    visium = sc.read_h5ad(visium_path)
    visium = preprocess_data(visium)
    if 'counts' in visium.layers:
        visium.X = visium.layers['counts'].astype(np.int32).copy()
    else:
        visium.X = np.round(visium.X).astype(np.int32)

    sc_data = sc.read_h5ad(sc_path)
    sc_data = preprocess_data(sc_data)
    if 'raw_counts' in sc_data.layers:
        sc_data.X = sc_data.layers['raw_counts'].astype(np.int32).copy()
    else:
        sc_data.X = np.round(sc_data.X).astype(np.int32)

    batches = sc_data.obs['batch'].astype(str).values
    sc_data.obs['batch'] = pd.Categorical(batches, categories=np.unique(batches), ordered=False)

    visium_uninfected = visium[visium.obs['condition'] == 'uninfected'].copy()
    visium_infected = visium[visium.obs['condition'] == 'infected'].copy()
    sc_uninfected = sc_data[sc_data.obs['timepoint'] == '0wk'].copy()
    sc_infected = sc_data[sc_data.obs['timepoint'] == '3wk'].copy()

    for subset in [sc_uninfected, sc_infected]:
        b = subset.obs['batch'].astype(str).values
        subset.obs['batch'] = pd.Categorical(b, categories=np.unique(b), ordered=False)

    return visium_uninfected, visium_infected, sc_uninfected, sc_infected


def run_fine_deconvolution(visium_data, sc_data, condition_name):
    cell2location.models.RegressionModel.setup_anndata(
        sc_data, batch_key='batch', labels_key='celltypes',
    )

    ref_model = cell2location.models.RegressionModel(sc_data)
    ref_model.train(
        max_epochs=10000,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        callbacks=[EarlyStopping(monitor="elbo_train", min_delta=0.001, patience=100, mode="min")]
    )
    sc_data = ref_model.export_posterior(sc_data)

    common_genes = visium_data.var_names.intersection(sc_data.var_names)
    visium_data = visium_data[:, common_genes].copy()
    sc_data = sc_data[:, common_genes].copy()

    cell2location.models.Cell2location.setup_anndata(visium_data)
    model = cell2location.models.Cell2location(
        visium_data,
        cell_state_df=sc_data.varm['means_per_cluster_mu_fg'].loc[visium_data.var_names],
        N_cells_per_location=30,
        detection_alpha=20,
    )

    model.train(
        max_epochs=30000,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        callbacks=[EarlyStopping(monitor="elbo_train", min_delta=0.001, patience=200, mode="min")],
        plan_kwargs={
            'optim': pyro.optim.Adamax({'lr': 0.01, 'weight_decay': 0.01}),
            'scale_elbo': 1.0
        }
    )

    visium_data = model.export_posterior(visium_data)

    visium_data.obsm['q05_cell_abundance_w_sf'].columns = (
        visium_data.obsm['q05_cell_abundance_w_sf'].columns
        .str.replace('q05cell_abundance_w_sf_means_per_cluster_mu_fg_', '')
    )
    visium_data.obsm['means_cell_abundance_w_sf'].columns = (
        visium_data.obsm['means_cell_abundance_w_sf'].columns
        .str.replace('meanscell_abundance_w_sf_means_per_cluster_mu_fg_', '')
    )

    visium_data.obs['sample'] = visium_data.obs.index.str.split('_').str[4]
    visium_data.obs['condition'] = condition_name
    return visium_data


def resolve_gene_expression(visium_data, sc_data, threshold=0.01):
    common_genes = visium_data.var_names.intersection(sc_data.var_names)
    print(f"Common genes: {len(common_genes)}")

    visium_subset = visium_data[:, common_genes].copy()
    sc_subset = sc_data[:, common_genes].copy()

    fine_proportions = pd.DataFrame(
        visium_data.obsm['q05_cell_abundance_w_sf'], index=visium_data.obs_names
    )
    name_mapping = {col: col.replace('q05cell_abundance_w_sf_means_per_cluster_mu_fg_', '')
                    for col in fine_proportions.columns}

    fine_expression = np.zeros((visium_subset.n_obs, len(common_genes)))

    for spot_idx in range(visium_subset.n_obs):
        spot_expression = np.zeros((1, len(common_genes)))
        for deconv_name, proportion in fine_proportions.iloc[spot_idx].items():
            if proportion > threshold:
                cell_type = name_mapping[deconv_name]
                cluster_cells = sc_subset[sc_subset.obs['celltypes'] == cell_type]
                if len(cluster_cells) == 0:
                    print(f"Warning: No cells found for cluster {cell_type}")
                    continue
                cluster_expression = cluster_cells.X.mean(axis=0)
                if scipy.sparse.issparse(cluster_expression):
                    cluster_expression = cluster_expression.toarray()
                cluster_expression = np.asarray(cluster_expression).reshape(1, -1)
                spot_expression += cluster_expression * proportion
        fine_expression[spot_idx] = spot_expression

    if np.isnan(fine_expression).any():
        raise ValueError("NaN values found in resolved gene expression")

    visium_data.layers['fine_expression'] = scipy.sparse.csr_matrix(np.zeros((visium_data.n_obs, visium_data.n_vars)))
    common_gene_indices = [list(visium_data.var_names).index(gene) for gene in common_genes]
    visium_data.layers['fine_expression'][:, common_gene_indices] = fine_expression
    return visium_data


def plot_fine_clusters(adata, condition_name, slice_to_plot, output_dir="./fine_cluster_plots_30000"):
    output_dir = Path(output_dir) / condition_name
    output_dir.mkdir(exist_ok=True, parents=True)

    slice_data = adata[adata.obs['sample'] == slice_to_plot].copy()
    print(f"\nPlotting {condition_name} - {slice_to_plot}")

    cell_types = slice_data.obsm['means_cell_abundance_w_sf'].columns
    print(f"Found {len(cell_types)} cell types")

    for cell_type in cell_types:
        try:
            slice_data.obs[cell_type] = slice_data.obsm['means_cell_abundance_w_sf'][cell_type]
            with plt.rc_context({'figure.figsize': (15, 15)}):
                plot_spatial(
                    adata=slice_data, color=[cell_type], labels=[cell_type],
                    show_img=True, style='fast', max_color_quantile=0.992,
                    circle_diameter=6, colorbar_position='right'
                )
                safe_name = cell_type.replace('/', '-')
                plt.savefig(output_dir / f"{slice_to_plot}_{safe_name}.png",
                            dpi=300, bbox_inches='tight', facecolor='white')
                plt.close()
            del slice_data.obs[cell_type]
        except Exception as e:
            print(f"Error plotting {cell_type}: {str(e)}")
            plt.close()

    print(f"Individual fine cluster plots saved to {output_dir}")


def main():
    visium_path = "/home/robeylab/cellxgene_data/1_18_25_visium_annotated.h5ad"
    sc_path = "/home/robeylab/cellxgene_data/preprocessed_deconvolution_sc_spleen_240116.h5ad"

    print("Loading and splitting data...")
    visium_uninfected, visium_infected, sc_uninfected, sc_infected = load_and_split_data(visium_path, sc_path)

    print("\nRunning deconvolution for uninfected samples...")
    visium_uninfected = run_fine_deconvolution(visium_uninfected, sc_uninfected, "uninfected")
    print("\nRunning deconvolution for infected samples...")
    visium_infected = run_fine_deconvolution(visium_infected, sc_infected, "infected")

    print("\nResolving gene expression...")
    visium_uninfected = resolve_gene_expression(visium_uninfected, sc_uninfected)
    visium_infected = resolve_gene_expression(visium_infected, sc_infected)

    print("\nGenerating plots...")
    for s in ['V1S1', 'V1S2']:
        plot_fine_clusters(visium_infected, "infected", slice_to_plot=s)
    for s in ['V1S3', 'V1S4']:
        plot_fine_clusters(visium_uninfected, "uninfected", slice_to_plot=s)

    print("\nSaving results...")
    visium_uninfected.write("uninfected_fine_30000.h5ad")
    visium_infected.write("infected_fine_30000.h5ad")
    print("\nAnalysis complete!")

    import gc
    gc.collect()

if __name__ == "__main__":
    main()

/usr/local/share/spleen_visium/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
2025-03-05 13:46:12.349519: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-05 13:46:12.357579: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-05 13:46:12.360036: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has alr

Using GPU: NVIDIA GeForce RTX 4090
Loading and splitting data...

Running deconvolution for uninfected samples...


<frozen abc>:119: FutureWarning: SparseDataset is deprecated and will be removed in late 2024. It has been replaced by the public classes CSRDataset and CSCDataset.

For instance checks, use `isinstance(X, (anndata.experimental.CSRDataset, anndata.experimental.CSCDataset))` instead.

For creation, use `anndata.experimental.sparse_dataset(X)` instead.

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/usr/local/share/spleen_visium/lib/python3.12/site-packages/lightning/pytorch/trainer/configuration_validator.py:72: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch

Epoch 10000/10000: 100%|██████████| 10000/10000 [1:11:01<00:00,  2.21it/s, v_num=1, elbo_train=5.31e+7]

`Trainer.fit` stopped: `max_epochs=10000` reached.


...

Sampling global variables, sample: 100%|██████████| 999/999 [00:04<00:00, 219.79it/s]

Resolving gene expression...
Common genes: 5155


/usr/local/share/spleen_visium/lib/python3.12/site-packages/scipy/sparse/_index.py:143: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)


Common genes: 5155


/usr/local/share/spleen_visium/lib/python3.12/site-packages/scipy/sparse/_index.py:143: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)



Generating plots...

Plotting infected - V1S1
Found 45 cell types
Individual fine cluster plots saved to fine_cluster_plots_30000/infected

Plotting infected - V1S2
Found 45 cell types
Individual fine cluster plots saved to fine_cluster_plots_30000/infected

Plotting uninfected - V1S3
Found 45 cell types
Individual fine cluster plots saved to fine_cluster_plots_30000/uninfected

Plotting uninfected - V1S4
Found 45 cell types
Individual fine cluster plots saved to fine_cluster_plots_30000/uninfected

Saving results...

Analysis complete! Results saved.

Cleaning up memory...
